# Trabajo Práctico N° 1: Cálculo Dosimétrico Manual con $^{177}\text{Lu}$-DOTATATE
**Curso:** Herramientas Computacionales en Dosimetría Interna (2026)  
**Paciente A:** Adulto, varón | Actividad administrada $A_0 = 7,21\text{ GBq} = 7210\text{ MBq}$

* **Masa Riñón Derecho ($m_{RD}$):** $230,877\text{ g}$
* **Masa Riñón Izquierdo ($m_{RI}$):** $237,007\text{ g}$
* **Masa Total Riñones ($m_{\text{paciente}}$):** $467,884\text{ g}$ (vs. fantoma ICRP $m_{\text{ref}} = 422\text{ g}$)
* **Masa Lesión Hepática:** $57\text{ g}$ ($0,057\text{ kg}$)
* **Período de semidesintegración físico ($^{177}\text{Lu}$):** $T_{1/2} = 6,65\text{ días} = 159,6\text{ h}$ ($\lambda_{\text{fís}} = \ln(2)/159,6 \approx 0,004343\text{ h}^{-1}$)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Configuración visual de gráficos
plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.4

# 1. Datos experimentales cuantificados por SPECT/CT (corregidos por decaimiento físico)
datos = {
    'time_h': [3.74111, 27.7253, 103.094, 123.741],
    'Bazo_MBq': [76.4439, 79.1639, 37.3715, 25.1662],
    'Rinon_D_MBq': [89.9996, 68.4121, 22.2228, 15.5858],
    'Rinon_I_MBq': [94.2604, 74.3360, 24.9881, 15.7949],
    'Higado_MBq': [263.672, 205.267, 107.993, 92.000],
    'Lesion_hepatica_MBq': [123.97, 142.28, 87.30, 76.79]
}

df = pd.DataFrame(datos)

# Parámetros físicos y anatómicos del paciente
A0_MBq = 7210.0           # 7.21 GBq
T_half_h = 6.65 * 24      # 159.6 horas
lambda_fis = np.log(2) / T_half_h  # ~0.004343 h^-1

masa_RD = 230.877         # g
masa_RI = 237.007         # g
masa_rinones_pac = masa_RD + masa_RI  # 467.884 g
masa_rinones_ref = 422.0  # g (fantoma ICRP 110 AM con sangre)
masa_lesion = 57.0        # g

df

## Parte A: Cálculo dosimétrico en riñones (autoirradiación)
1. **Gráfico de retención** de Riñón Derecho ($RD$) y Riñón Izquierdo ($RI$).
2. **Área bajo la curva ($\tilde{A}$):**
   * **2a)** Trapezoide sin corrección (entre $t_1$ y $t_4$).
   * **2b)** Trapezoide + corrección lineal terminal (recta por los 2 últimos puntos hasta $A=0$).
   * **2c)** Trapezoide + corrección exponencial terminal (decaimiento físico puro desde $t_4 \to \infty$).
   * **2d)** Ajuste por cuadrados mínimos monoexponencial $A(t) = A_0 e^{-\lambda_{\text{ef}} t}$ integrado de $0 \to \infty$.

In [ ]:
t = df['time_h'].values
A_RD = df['Rinon_D_MBq'].values
A_RI = df['Rinon_I_MBq'].values

# --- 1. Gráfico de retención ---
plt.figure()
plt.plot(t, A_RD, 'o-', label='Riñón Derecho (MBq)', linewidth=2)
plt.plot(t, A_RI, 's-', label='Riñón Izquierdo (MBq)', linewidth=2)
plt.title('Curva de Actividad vs Tiempo en Riñones')
plt.xlabel('Tiempo post-administración (h)')
plt.ylabel('Actividad (MBq)')
plt.legend()
plt.show()

# Función auxiliar para calcular los 4 métodos de AUC
def calcular_auc_rinon(t, A, nombre_organo):
    # 2a) Trapezoide sin corrección (t1 a t4)
    auc_trap = np.trapezoid(A, t)
    
    # 2b) Corrección lineal con los últimos 2 puntos (t3, A3) y (t4, A4)
    pendiente = (A[-1] - A[-2]) / (t[-1] - t[-2])
    ordenada = A[-1] - pendiente * t[-1]
    t_corte = -ordenada / pendiente  # tiempo donde A(t) = 0
    area_triangulo = 0.5 * (t_corte - t[-1]) * A[-1]
    auc_lin = auc_trap + area_triangulo
    
    # 2c) Corrección exponencial (decaimiento físico desde t4 a infinito)
    area_cola_exp = A[-1] / lambda_fis
    auc_exp = auc_trap + area_cola_exp
    
    # 2d) Ajuste por cuadrados mínimos monoexponencial: A(t) = a * exp(-b * t)
    func_mono = lambda x, a, b: a * np.exp(-b * x)
    popt, _ = curve_fit(func_mono, t, A, p0=[A[0], 0.015])
    a_fit, b_fit = popt
    auc_lsq = a_fit / b_fit  # Integral analítica de 0 a infinito
    
    print(f'=== {nombre_organo} ===')
    print(f'2a) Trapezoide sin corrección:          {auc_trap:.1f} MBq·h')
    print(f'2b) Trapezoide + cola lineal:           {auc_lin:.1f} MBq·h (t_corte = {t_corte:.2f} h)')
    print(f'2c) Trapezoide + cola exponencial fís.: {auc_exp:.1f} MBq·h (cola = {area_cola_exp:.1f} MBq·h)')
    print(f'2d) Ajuste Mínimos Cuadrados (0->inf):  {auc_lsq:.1f} MBq·h  [A(t) = {a_fit:.2f} * exp(-{b_fit:.5f}*t)]\n')
    
    return {'trap': auc_trap, 'lin': auc_lin, 'exp': auc_exp, 'lsq': auc_lsq, 'popt': popt}

res_RD = calcular_auc_rinon(t, A_RD, 'RIÑÓN DERECHO')
res_RI = calcular_auc_rinon(t, A_RI, 'RIÑÓN IZQUIERDO')

# Gráfico comparativo del ajuste monoexponencial (2d)
t_fino = np.linspace(0, 145, 200)
plt.figure()
plt.scatter(t, A_RD, color='tab:blue', label='Datos Riñón D')
plt.plot(t_fino, res_RD['popt'][0] * np.exp(-res_RD['popt'][1] * t_fino), '--', color='tab:blue',
         label=f"Ajuste RD: {res_RD['popt'][0]:.2f}·exp(-{res_RD['popt'][1]:.4f}t)")

plt.scatter(t, A_RI, color='tab:orange', label='Datos Riñón I')
plt.plot(t_fino, res_RI['popt'][0] * np.exp(-res_RI['popt'][1] * t_fino), '--', color='tab:orange',
         label=f"Ajuste RI: {res_RI['popt'][0]:.2f}·exp(-{res_RI['popt'][1]:.4f}t)")

plt.title('Ajuste Monoexponencial por Cuadrados Mínimos (0 a ∞)')
plt.xlabel('Tiempo (h)')
plt.ylabel('Actividad (MBq)')
plt.legend()
plt.show()

### Paso 3: Cálculo dosimétrico con Factores $S$ (MIRDsoft vs. OpenDose)
* **MIRDsoft (ICRP 110 AM):** $S_{\beta^-} = 5,57 \times 10^{-5}\text{ mGy/(MBq}\cdot\text{s)}$ (o $S_{\text{total}} = 5,69 \times 10^{-5}\text{ mGy/(MBq}\cdot\text{s)}$).
* **OpenDose (ICRP 110 AM, RKidney + LKidney):** $S_{\text{total}} = 7,72 \times 10^{-5}\text{ mGy/(MBq}\cdot\text{s)}$.
* **Ajuste de masa (autoirradiación):** $S_{\text{paciente}} = S_{\text{ref}} \cdot \left(\frac{m_{\text{ref}}}{m_{\text{paciente}}}\right)$ (escala lineal $1/m$ para partículas $\beta^-$, aunque en la guía también se muestra el factor $(422/467,884)^{1/3} = 0,966$ por efecto de superficie/alcance). Calculamos ambos para comparar.

In [ ]:
# Actividad acumulada total en ambos riñones (usando ajuste cuadrados mínimos 2d)
A_tilde_total_rinones = res_RD['lsq'] + res_RI['lsq']  # MBq·h
TIAC_total_rinones = A_tilde_total_rinones / A0_MBq      # horas (dato útil para TP 2 en IDAC-Dose)

print(f'Actividad acumulada total (RD + RI): {A_tilde_total_rinones:.1f} MBq·h')
print(f'TIAC total riñones (ã):              {TIAC_total_rinones:.4f} h\n')

# Factores S tabulados [mGy / (MBq·s)]
S_MIRD_beta = 5.57e-5
S_MIRD_tot  = 5.69e-5
S_OpenDose  = 7.72e-5

# Factor de escala de masa utilizado en la guía resuelta: (m_ref / m_pac)^(1/3) = 0.966
# Y factor de escala lineal estándar MIRD para electrones/beta: (m_ref / m_pac)^1 = 0.9019
factor_guia = (masa_rinones_ref / masa_rinones_pac) ** (1/3)
factor_lineal = (masa_rinones_ref / masa_rinones_pac)

def calcular_dosis_rinones(S_tab, factor_escala, etiqueta):
    S_adj = S_tab * factor_escala
    # Multiplicamos por 3600 s/h para convertir MBq·h a MBq·s
    D_mGy = A_tilde_total_rinones * 3600 * S_adj
    D_Gy = D_mGy / 1000.0
    print(f'{etiqueta:<42} | S_adj = {S_adj:.3e} mGy/MBq·s | D = {D_mGy:6.1f} mGy ({D_Gy:.2f} Gy)')

print('--- Resultados de Autoirradiación en Ambos Riñones ---')
calcular_dosis_rinones(S_MIRD_beta, factor_guia,   'MIRDsoft (beta-, factor guía ^1/3)')
calcular_dosis_rinones(S_MIRD_tot,  factor_guia,   'MIRDsoft (total, factor guía ^1/3)')
calcular_dosis_rinones(S_OpenDose,  factor_guia,   'OpenDose (total, factor guía ^1/3)')
print('-' * 92)
calcular_dosis_rinones(S_MIRD_tot,  factor_lineal, 'MIRDsoft (total, escala lineal 1/m)')
calcular_dosis_rinones(S_OpenDose,  factor_lineal, 'OpenDose (total, escala lineal 1/m)')

## Parte B: Cálculo dosimétrico en lesión tumoral hepática ($m = 57\text{ g}$)
1. **Ajuste monoexponencial** de la curva tiempo-actividad e integración de $0 \to \infty$.
2. **Cálculo de $S_{\text{esfera}}$ (Stabin & Konijnenberg, 2000):**
   * Emisión $\beta^-$ principal ($^{177}\text{Lu}$): $y_\beta = 0,79$, $E_\beta = 133\text{ keV}$, fracción absorbida (esfera $60\text{ g}$, $0,1\text{ MeV}$) $\phi_\beta = 0,997$.
   * Emisión $\gamma$ principal ($^{177}\text{Lu}$): $y_\gamma = 0,11$, $E_\gamma = 208\text{ keV}$, fracción absorbida (esfera $60\text{ g}$, $0,14\text{ MeV}$) $\phi_\gamma = 0,053$.
3. **Dosis absorbida tumoral:** $D_{\text{tumor}} = \tilde{A}_{\text{tumor}} \cdot S_{\text{esfera}}$.

In [ ]:
A_lesion = df['Lesion_hepatica_MBq'].values

# 1. Ajuste monoexponencial: A(t) = a * exp(-b * t)
func_mono = lambda x, a, b: a * np.exp(-b * x)
popt_les, _ = curve_fit(func_mono, t, A_lesion, p0=[140, 0.005])
a_les, b_les = popt_les
A_tilde_lesion = a_les / b_les  # MBq·h

# Gráfico con área sombreada bajo la curva
t_ext = np.linspace(0, 150, 300)
y_ext = func_mono(t_ext, a_les, b_les)

plt.figure()
plt.scatter(t, A_lesion, color='red', s=60, zorder=5, label='Datos SPECT/CT Lesión')
plt.plot(t_ext, y_ext, 'b-', linewidth=2, label=f'Ajuste: {a_les:.2f} · exp(-{b_les:.4f}·t)')
plt.fill_between(t_ext, y_ext, color='skyblue', alpha=0.35, label=f'AUC (0-∞) = {A_tilde_lesion:.1f} MBq·h')
plt.title('Curva Tiempo-Actividad en Lesión Hepática (57 g)')
plt.xlabel('Tiempo (h)')
plt.ylabel('Actividad (MBq)')
plt.xlim(0, 150)
plt.ylim(0, 155)
plt.legend()
plt.show()

# 2. Cálculo del Factor S de esfera (Stabin & Konijnenberg, 2000)
y_beta, E_beta_keV, phi_beta = 0.79, 133.0, 0.997
y_gamma, E_gamma_keV, phi_gamma = 0.11, 208.0, 0.053
m_tumor_kg = masa_lesion / 1000.0  # 0.057 kg

# Energía absorbida por unidad de masa por desintegración [keV / (kg·desint)]
energia_por_kg_keV = (y_beta * E_beta_keV * phi_beta + y_gamma * E_gamma_keV * phi_gamma) / m_tumor_kg

# Conversión de unidades: 1 keV = 1.60218e-16 J ; 1 Gy = 1 J/kg
keV_a_J = 1.60218e-16
S_esfera_Gy_des = energia_por_kg_keV * keV_a_J              # Gy / desintegración
S_esfera_mGy_MBqs = S_esfera_Gy_des * 1e6 * 1e3            # mGy / (MBq·s)
S_esfera_mGy_MBqh = S_esfera_mGy_MBqs * 3600               # mGy / (MBq·h)

# 3. Dosis absorbida en la lesión hepática
D_tumor_mGy = A_tilde_lesion * S_esfera_mGy_MBqh
D_tumor_Gy = D_tumor_mGy / 1000.0

print('=== RESULTADOS LESIÓN HEPÁTICA (57 g) ===')
print(f'Actividad acumulada (Ã): {A_tilde_lesion:.2f} MBq·h')
print(f'Energía absorbida/masa:  {energia_por_kg_keV:.1f} keV/kg')
print(f'S_esfera:                {S_esfera_mGy_MBqs:.3e} mGy/(MBq·s) = {S_esfera_mGy_MBqh:.4f} mGy/(MBq·h)')
print(f'Dosis Absorbida Tumor:   {D_tumor_mGy:.1f} mGy = {D_tumor_Gy:.2f} Gy')

### Notas para el informe y continuidad con el TP N° 2
* **Intervalo $0 - 3,74\text{ h}$:** Al integrar por trapezoides desde $t_1 = 3,74\text{ h}$, se omite la fase inicial de captación. En la práctica clínica suele asumirse actividad inicial nula en el órgano $A(0) = 0$ (sumando el triángulo $\frac{1}{2} A_1 t_1$) o extrapolar hacia atrás la curva exponencial ajustada hasta $t = 0$.
* **Datos listos para IDAC-Dose (TP 2):** El TIAC total de riñones ($\tilde{a} \approx 2,00\text{ h}$) y la actividad acumulada de la lesión hepática ($\tilde{A}_{\text{tumor}} \approx 31\,825\text{ MBq}\cdot\text{h}$) son los valores de entrada directos para auditar el cálculo en IDAC-Dose e IDAC Spheres.